In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, f1_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_score



print("""
PREDICTING STUDENT DISENGAGEMENT IN AR-BASED STEM LEARNING
Marko Mikec Bachelor Thesis Code


Three files needed for this script:
1. STEMGeometry.csv              - xAPI interaction logs (geometry)
2. STEMGeography.csv             - xAPI interaction logs (geography)

3. 20230118_P2 Students Test_share_ids_info.xlsx — TIMSS pre/post test scores 
(ecological validation ONLY, not used for clustering or ML)

Random seed 42 fixed throughout therefore results are fully reproducible.

Figures produced:

  fig_elbow_geometry.png / fig_elbow_geography.png
  fig_gain_geometry.png / fig_gain_geography.png
  fig_importance_geometry.png / fig_importance_geography.png  
  fig_cluster_scatter_geometry.png / fig_cluster_scatter_geography.png   
  fig_cross_subject.png  
  fig_roc_combined.png               
  fig_separation_boxplots.png        
  fig_cluster_two_feature.png 

Outputs will be saved to new file outputs within folder!
""")


warnings.filterwarnings('ignore')
np.random.seed(42)
os.makedirs('outputs', exist_ok=True)

# Clean style for the figures
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
})

# Paths
GEOMETRY_FILE = 'STEMGeometry.csv'
GEOGRAPHY_FILE = 'STEMGeography.csv'
TEST_FILE = '20230118_P2 Students Test_share_ids_info.xlsx'

# Color changes
COLOR_GEOM = 'darkorange'    # Geometry throughout
COLOR_GEO = 'mediumseagreen'  # Geography throughout

# Pilot 2 verbs
KNOWN_VERBS = {'launched', 'answered', 'interacted', 'selected', 'completed', 'exited'}
INTERACTION_VERBS = {'interacted', 'selected'}

# Features
FEATURE_COLS = [
    'session_count',     
    'total_dur_min',      
    'avg_session_dur',    
    'total_actions',      
    'interaction_rate',   
    'verb_diversity',     
    'completion_count',   
    'completion_rate',    
    'answered_count',     
]

RANDOM_STATE = 42
N_SPLITS = 5

In [18]:
# XAPI logs input
print("[Loading xAPI logs]")

def parse_verb(raw_verb):
    if pd.isna(raw_verb):
        return 'unknown'
    s = str(raw_verb).lower()
    for v in KNOWN_VERBS:
        if v in s:
            return v
    return 'other'


def parse_score(raw_result):
    if pd.isna(raw_result):
        return np.nan
    m = re.search(r'"raw"\s*:\s*([\d.]+)', str(raw_result))
    return float(m.group(1)) if m else np.nan


def load_xapi(filepath, subject_label):
    df = pd.read_csv(filepath)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    if 'actor_name' in df.columns:
        df = df.rename(columns={'actor_name': 'actor'})
    df['timestamp']  = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    df['verb_clean'] = df['verb_display'].apply(parse_verb)
    df['score_raw']  = df['result'].apply(parse_score)
    df['subject']    = subject_label
    df = df.dropna(subset=['actor', 'timestamp'])

    print(f"\n{subject_label}:")
    print(f"  Rows:       {len(df):,}")
    print(f"  Students:   {df['actor'].nunique():,}")
    print(f"  Date range: {df['timestamp'].min().date()} — {df['timestamp'].max().date()}")
    print(f"  Verb counts:\n{df['verb_clean'].value_counts().to_string()}")
    return df


geom_df = load_xapi(GEOMETRY_FILE,  'geometry')
geo_df = load_xapi(GEOGRAPHY_FILE, 'geography')
print("\nxAPI logs loaded")

[Loading xAPI logs]

geometry:
  Rows:       142,567
  Students:   694
  Date range: 2021-10-15 — 2022-07-14
  Verb counts:
verb_clean
selected      59354
answered      53608
interacted    14016
launched       5868
completed      5564
other          4157

geography:
  Rows:       136,416
  Students:   739
  Date range: 2021-10-15 — 2022-07-05
  Verb counts:
verb_clean
interacted    60636
selected      43315
launched      23051
answered       8689
completed       725

xAPI logs loaded


In [22]:
# Feature Engineering
print("[Feature engineering (one row per student)]")

def engineer_features(df, subject_label):
    records = []
    for actor, grp in df.groupby('actor'):
        grp = grp.sort_values('timestamp')
        n = len(grp)

        #Temporal (Calendar dates used as session proxy, lrs_id is shared)
        grp = grp.copy()
        grp['date'] = grp['timestamp'].dt.date
        sessions  = grp['date'].nunique()
        total_dur = (grp['timestamp'].max() - grp['timestamp'].min()).total_seconds() / 60.0
        sess_durs = []
        for _, sg in grp.groupby('date'):
            d = (sg['timestamp'].max() - sg['timestamp'].min()).total_seconds() / 60.0
            if d >= 1.0:
                sess_durs.append(d)
        avg_sess = float(np.mean(sess_durs)) if sess_durs else 0.0

        # Behavioral
        interact_n = grp['verb_clean'].isin(INTERACTION_VERBS).sum()
        verb_div = grp['verb_clean'].nunique()

        # Progress
        comp_n = (grp['verb_clean'] == 'completed').sum()
        answered_n = (grp['verb_clean'] == 'answered').sum()
        scores = grp['score_raw'].dropna()
        mean_score = scores.mean() if len(scores) > 0 else np.nan

        records.append({
            'actor' : actor,
            'session_count' : sessions,
            'total_dur_min' : total_dur,
            'avg_session_dur' : avg_sess,
            'total_actions' : n,
            'interaction_rate' : interact_n / n if n > 0 else 0.0,
            'verb_diversity' : verb_div,
            'completion_count': comp_n,
            'completion_rate' : comp_n / n if n > 0 else 0.0,
            'answered_count' : answered_n,
            'mean_score' : mean_score,
        })

    feat = pd.DataFrame(records)
    feat = feat[feat['total_actions'] > 1].reset_index(drop=True)

    print(f"\n{subject_label}: {len(feat)} students after filtering single-event actors")
    print(feat[FEATURE_COLS].describe().round(2).to_string())
    feat.to_csv(f'outputs/features_{subject_label}.csv', index=False)
    return feat


feat_geom = engineer_features(geom_df, 'geometry')
feat_geo = engineer_features(geo_df,  'geography')
print("\nFeature CSV complete")

[Feature engineering (one row per student)]

geometry: 677 students after filtering single-event actors
       session_count  total_dur_min  avg_session_dur  total_actions  interaction_rate  verb_diversity  completion_count  completion_rate  answered_count
count         677.00         677.00           677.00         677.00            677.00          677.00            677.00           677.00          677.00
mean            3.03       32023.55            34.88         210.56              0.59            3.93              8.22             0.03           79.18
std             2.72       51442.32            55.59         267.48              0.34            1.58             19.23             0.03          186.77
min             1.00           0.09             0.00           2.00              0.00            1.00              0.00             0.00            0.00
25%             1.00          18.61            11.30          46.00              0.26            3.00              0.00            

In [26]:
# External Testing
print("[External test scores]")

def load_test_scores(filepath):
    raw  = pd.read_excel(filepath, sheet_name='Raw Data')
    MISS = [8, '8a', 9, '9a']

    def score_cols(df, cols):
        return df[cols].replace(MISS, np.nan).apply(pd.to_numeric, errors='coerce')

    m_pre = [c for c in raw.columns if c[:2] in ('M1','M2','M3')
              and '_post' not in c and '_ret' not in c]
    m_post = [c for c in raw.columns if c[:2] in ('M1','M2','M3') and '_post' in c]
    s_pre = [c for c in raw.columns
              if (c.startswith('S1_') or c.startswith('S2_'))
              and '_post' not in c and '_ret' not in c]
    s_post = [c for c in raw.columns
              if (c.startswith('S1_') or c.startswith('S2_')) and '_post' in c]

    out = {}
    for subj_code, label, pre_c, post_c in [
        (1, 'geometry',  m_pre, m_post),
        (2, 'geography', s_pre, s_post),
    ]:
        sub = raw[raw['SUBJ'] == subj_code].copy()
        pre  = score_cols(sub, pre_c)
        post = score_cols(sub, post_c)
        sub = sub.copy()
        sub['pre_score'] = pre.sum(axis=1, skipna=True)
        sub['post_score'] = post.sum(axis=1, skipna=True)
        sub['pre_pct'] = sub['pre_score']  / len(pre_c)  * 100
        sub['post_pct'] = sub['post_score'] / len(post_c) * 100
        sub['gain']  = sub['post_score'] - sub['pre_score']
        keep = sub[['StudentID','GRP','pre_score','post_score',
                    'pre_pct','post_pct','gain']].copy()
        out[label] = keep
        keep.to_csv(f'outputs/test_scores_{label}.csv', index=False)

        int_gain  = keep[keep['GRP']==1]['gain'].dropna()
        ctrl_gain = keep[keep['GRP']==2]['gain'].dropna()
        u, p = stats.mannwhitneyu(int_gain, ctrl_gain, alternative='greater')
        print(f"\n{label} (n={len(keep)}):")
        print(f"  Intervention: pre={keep[keep['GRP']==1]['pre_pct'].mean():.1f}%, "
              f"post={keep[keep['GRP']==1]['post_pct'].mean():.1f}%, "
              f"gain={int_gain.mean():.2f}")
        print(f"  Control:      pre={keep[keep['GRP']==2]['pre_pct'].mean():.1f}%, "
              f"post={keep[keep['GRP']==2]['post_pct'].mean():.1f}%, "
              f"gain={ctrl_gain.mean():.2f}")
        print(f"  Mann-Whitney (int > ctrl): U={u:.0f}, p={p:.4f}")

    return out['geometry'], out['geography']


scores_geom, scores_geo = load_test_scores(TEST_FILE)
print("\nTest scores complete")

[External test scores]

geometry (n=1047):
  Intervention: pre=21.0%, post=25.2%, gain=1.33
  Control:      pre=22.1%, post=24.8%, gain=0.87
  Mann-Whitney (int > ctrl): U=149362, p=0.0015

geography (n=941):
  Intervention: pre=33.1%, post=38.3%, gain=1.30
  Control:      pre=33.3%, post=37.7%, gain=1.10
  Mann-Whitney (int > ctrl): U=111700, p=0.2203

Test scores complete


In [28]:
# K-means clustering
print("[K-means clustering]")

def cluster_subject(feat, label, color):
    feat = feat.copy()
    for col in FEATURE_COLS:
        feat[col] = feat[col].fillna(feat[col].median())

    X = feat[FEATURE_COLS].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Elbow and silhouette
    ks, inertias, silhouettes = range(2, 9), [], []
    for k in ks:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        km.fit(X_scaled)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X_scaled, km.labels_))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(list(ks), inertias,    'o-', color=color, linewidth=2)
    axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
    axes[0].set_title(f'{label}: Elbow plot'); axes[0].set_xticks(list(ks))
    axes[1].plot(list(ks), silhouettes, 's-', color=color, linewidth=2)
    axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette score')
    axes[1].set_title(f'{label}: Silhouette scores'); axes[1].set_xticks(list(ks))
    plt.tight_layout()
    plt.savefig(f'outputs/fig_elbow_{label}.png', bbox_inches='tight')
    plt.close()

    print(f"\n{label} — silhouette by k:")
    for k, s in zip(ks, silhouettes):
        print(f"  k={k}: {s:.3f}")

    # Fit k=2
    km2 = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
    feat['cluster'] = km2.fit_predict(X_scaled)
    sil2 = silhouette_score(X_scaled, feat['cluster'])
    disengaged_cluster = feat.groupby('cluster')['total_actions'].mean().idxmin()
    feat['engagement'] = feat['cluster'].apply(
        lambda c: 'disengaged' if c == disengaged_cluster else 'engaged'
    )

    counts = feat['engagement'].value_counts()
    pct = counts.get('disengaged', 0) / len(feat) * 100
    print(f"\n{label} k=2 silhouette: {sil2:.3f}")
    print(f"  Engaged:    {counts.get('engaged', 0)}")
    print(f"  Disengaged: {counts.get('disengaged', 0)} ({pct:.1f}%)")

    profile = feat.groupby('engagement')[FEATURE_COLS].mean().round(2)
    profile.to_csv(f'outputs/cluster_profile_{label}.csv')
    print(f"\n  Cluster profile:\n{profile.to_string()}")

    
    return feat, X_scaled, scaler, sil2


feat_geom, X_geom, scaler_geom, sil_geom = cluster_subject(feat_geom, 'geometry',  COLOR_GEOM)
feat_geo,  X_geo,  scaler_geo,  sil_geo  = cluster_subject(feat_geo,  'geography', COLOR_GEO)
print("\nClustering complete")

[K-means clustering]


  File "C:\Users\marko\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\marko\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\marko\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\marko\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^



geometry — silhouette by k:
  k=2: 0.495
  k=3: 0.286
  k=4: 0.315
  k=5: 0.325
  k=6: 0.347
  k=7: 0.351
  k=8: 0.360

geometry k=2 silhouette: 0.495
  Engaged:    80
  Disengaged: 597 (88.2%)

  Cluster profile:
            session_count  total_dur_min  avg_session_dur  total_actions  interaction_rate  verb_diversity  completion_count  completion_rate  answered_count
engagement                                                                                                                                                   
disengaged           2.32       22290.37            32.67         136.64              0.63            3.73              3.13             0.02           29.45
engaged              8.38      104657.45            51.44         762.16              0.29            5.38             46.21             0.06          450.29

geography — silhouette by k:
  k=2: 0.477
  k=3: 0.307
  k=4: 0.309
  k=5: 0.329
  k=6: 0.295
  k=7: 0.302
  k=8: 0.307

geography k=2 silhouette: 0.477

In [30]:
# Ecological Validity
print("[Ecological validation]")

def validate(scores, label, color):
    g1 = scores[scores['GRP'] == 1]['gain'].dropna()
    g2 = scores[scores['GRP'] == 2]['gain'].dropna()

    print(f"\n{label}:")
    print(f"  Intervention (n={len(g1)}): mean gain={g1.mean():.2f}, SD={g1.std():.2f}")
    print(f"  Control      (n={len(g2)}): mean gain={g2.mean():.2f}, SD={g2.std():.2f}")

    u, p = stats.mannwhitneyu(g1, g2, alternative='greater')
    lev, levp = stats.levene(g1, g2)
    print(f"  Gain Mann-Whitney U={u:.0f}, p={p:.4f} (int > ctrl)")
    print(f"  Levene variance test W={lev:.2f}, p={levp:.4f}")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(g1, bins=20, alpha=0.65, label='Intervention', color=color,  density=True)
    ax.hist(g2, bins=20, alpha=0.40, label='Control',      color='grey', density=True)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Test score gain (post − pre)')
    ax.set_ylabel('Density')
    ax.set_title(f'{label}: Score gain distribution')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'outputs/fig_gain_{label}.png', bbox_inches='tight')
    plt.close()

    return {'label': label, 'int_mean': g1.mean(), 'ctrl_mean': g2.mean(),
            'int_sd': g1.std(), 'ctrl_sd': g2.std(),
            'U': u, 'p_gain': p, 'levene_p': levp}


v1 = validate(scores_geom, 'geometry',  COLOR_GEOM)
v2 = validate(scores_geo,  'geography', COLOR_GEO)
pd.DataFrame([v1, v2]).to_csv('outputs/validation_summary.csv', index=False)
print("\nValidation complete")

[Ecological validation]

geometry:
  Intervention (n=586): mean gain=1.33, SD=2.42
  Control      (n=461): mean gain=0.87, SD=2.22
  Gain Mann-Whitney U=149362, p=0.0015 (int > ctrl)
  Levene variance test W=4.26, p=0.0392

geography:
  Intervention (n=536): mean gain=1.30, SD=2.64
  Control      (n=405): mean gain=1.10, SD=2.84
  Gain Mann-Whitney U=111700, p=0.2203 (int > ctrl)
  Levene variance test W=2.78, p=0.0958

Validation complete


In [32]:
# Classification
print("[Classification (LR + RF, stratified 5-fold CV)]")

def classify(feat, X_scaled, label):
    y = (feat['engagement'] == 'disengaged').astype(int)
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    majority = int(y.mode()[0])
    bl_f1 = f1_score(y, [majority]*len(y), zero_division=0)
    bl_rec = recall_score(y, [majority]*len(y), zero_division=0)

    print(f"\n{label}  (n={len(y)}, disengaged={y.sum()}, {y.mean()*100:.1f}%)")
    print(f"  Baseline (always predict {majority}): F1={bl_f1:.3f}, recall={bl_rec:.3f}")

    rows = [{'model': 'Baseline', 'subject': label,
             'recall': bl_rec, 'recall_sd': 0,
             'f1': bl_f1, 'f1_sd': 0, 'auc': 0.5, 'auc_sd': 0}]

    rf_fitted = None
    for name, clf in [
        ('Logistic Regression',
         LogisticRegression(max_iter=1000, class_weight='balanced',
                            C=1.0, random_state=RANDOM_STATE)),
        ('Random Forest',
         RandomForestClassifier(n_estimators=200, max_depth=6,
                                class_weight='balanced',
                                random_state=RANDOM_STATE)),
    ]:
        rec = cross_val_score(clf, X_scaled, y, cv=cv, scoring='recall')
        f1 = cross_val_score(clf, X_scaled, y, cv=cv, scoring='f1')
        auc = cross_val_score(clf, X_scaled, y, cv=cv, scoring='roc_auc')

        print(f"\n  {name}:")
        print(f"    Recall : {rec.mean():.3f} ± {rec.std():.3f}")
        print(f"    F1     : {f1.mean():.3f}  ± {f1.std():.3f}")
        print(f"    AUC-ROC: {auc.mean():.3f} ± {auc.std():.3f}")
        rows.append({'model': name, 'subject': label,
                     'recall': rec.mean(), 'recall_sd': rec.std(),
                     'f1': f1.mean(), 'f1_sd': f1.std(),
                     'auc': auc.mean(), 'auc_sd': auc.std()})

        if name == 'Random Forest':
            clf.fit(X_scaled, y)
            rf_fitted = clf

    pd.DataFrame(rows).to_csv(f'outputs/performance_{label}.csv', index=False)
    return rf_fitted, y


rf_geom, y_geom = classify(feat_geom, X_geom, 'geometry')
rf_geo,  y_geo  = classify(feat_geo,  X_geo,  'geography')
print("\nClassification complete")

[Classification (LR + RF, stratified 5-fold CV)]

geometry  (n=677, disengaged=597, 88.2%)
  Baseline (always predict 1): F1=0.937, recall=1.000

  Logistic Regression:
    Recall : 0.990 ± 0.010
    F1     : 0.995  ± 0.005
    AUC-ROC: 1.000 ± 0.000

  Random Forest:
    Recall : 1.000 ± 0.000
    F1     : 0.996  ± 0.004
    AUC-ROC: 0.998 ± 0.003

geography  (n=725, disengaged=621, 85.7%)
  Baseline (always predict 1): F1=0.923, recall=1.000

  Logistic Regression:
    Recall : 0.989 ± 0.008
    F1     : 0.994  ± 0.004
    AUC-ROC: 1.000 ± 0.000

  Random Forest:
    Recall : 0.997 ± 0.004
    F1     : 0.990  ± 0.007
    AUC-ROC: 0.997 ± 0.004

Classification complete


In [33]:
# Permutation
print("[Permutation feature importance]")


def get_importance(rf, X_scaled, y, label, color, n_repeats=30):
    result = permutation_importance(
        rf, X_scaled, y,
        n_repeats=n_repeats,
        scoring='f1',
        random_state=RANDOM_STATE,
    )
    imp = pd.DataFrame({
        'feature' : FEATURE_COLS,
        'mean_imp': result.importances_mean,
        'std_imp' : result.importances_std,
    }).sort_values('mean_imp', ascending=False).reset_index(drop=True)

    print(f"\n{label}:")
    print(imp.to_string(index=False))
    imp.to_csv(f'outputs/importance_{label}.csv', index=False)

    pd.DataFrame(result.importances.T, columns=FEATURE_COLS).to_csv(
        f'outputs/importance_raw_{label}.csv', index=False
    )

    fig, ax = plt.subplots(figsize=(7, 5))
    yp = np.arange(len(imp))
    ax.barh(yp, imp['mean_imp'], xerr=imp['std_imp'],
            color=color, alpha=0.85, capsize=3)
    ax.set_yticks(yp)
    ax.set_yticklabels(imp['feature'])
    ax.invert_yaxis()
    ax.set_xlabel('Mean F1 decrease (permutation importance)')
    ax.set_title(f'{label}: Feature importance')
    plt.tight_layout()
    plt.savefig(f'outputs/fig_importance_{label}.png', bbox_inches='tight')
    plt.close()

    return imp, result.importances


imp_geom, raw_geom = get_importance(rf_geom, X_geom, y_geom, 'geometry',  COLOR_GEOM)
imp_geo,  raw_geo  = get_importance(rf_geo,  X_geo,  y_geo,  'geography', COLOR_GEO)
print("\nImportance computation complete")

[Permutation feature importance]

geometry:
         feature  mean_imp  std_imp
  answered_count  0.034723 0.002226
   session_count  0.009353 0.001363
   total_actions  0.007318 0.000917
   total_dur_min  0.003781 0.001068
completion_count  0.002561 0.000712
 completion_rate  0.000530 0.000403
interaction_rate  0.000418 0.000418
 avg_session_dur  0.000307 0.000403
  verb_diversity  0.000000 0.000000

geography:
         feature  mean_imp  std_imp
  answered_count  0.041998 0.002690
completion_count  0.025939 0.001562
   total_dur_min  0.012376 0.001948
   total_actions  0.006058 0.001086
 completion_rate  0.004170 0.000810
   session_count  0.001822 0.000461
interaction_rate  0.001287 0.000765
  verb_diversity  0.000456 0.000399
 avg_session_dur  0.000000 0.000000

Importance computation complete


In [36]:
# RQ2
print("[Cross-subject comparison]")

rows = []
for feat_idx, feat_name in enumerate(FEATURE_COLS):
    geom_vals = raw_geom[feat_idx]
    geo_vals = raw_geo[feat_idx]
    u, p = stats.mannwhitneyu(geom_vals, geo_vals, alternative='two-sided')
    rows.append({
        'feature' : feat_name,
        'geom_mean' : geom_vals.mean(),
        'geo_mean' : geo_vals.mean(),
        'difference' : geom_vals.mean() - geo_vals.mean(),
        'U' : u,
        'p_value' : p,
        'significant': p < 0.05,
    })

comp = pd.DataFrame(rows).sort_values('difference', key=abs, ascending=False)
comp.to_csv('outputs/cross_subject_comparison.csv', index=False)
print(comp.to_string(index=False))

# Cross-subject figure
comp_sorted = comp.sort_values('geom_mean', ascending=False)
x = np.arange(len(FEATURE_COLS))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, comp_sorted['geom_mean'], w, label='Geometry',  color=COLOR_GEOM, alpha=0.85)
ax.bar(x + w/2, comp_sorted['geo_mean'],  w, label='Geography', color=COLOR_GEO,  alpha=0.85)
for j, (_, row) in enumerate(comp_sorted.iterrows()):
    if row['significant']:
        ymax = max(row['geom_mean'], row['geo_mean'])
        ax.text(j, ymax + 0.002, '*', ha='center', fontsize=14, color='crimson')
ax.set_xticks(x)
ax.set_xticklabels(comp_sorted['feature'], rotation=35, ha='right')
ax.set_ylabel('Mean permutation importance (F1 drop)')
ax.set_title('Feature importance by subject   (* p < .05, Mann-Whitney U)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/fig_cross_subject.png', bbox_inches='tight')
plt.close()
print("\nCross-subject comparison complete")

[Cross-subject comparison]
         feature  geom_mean  geo_mean  difference     U      p_value  significant
completion_count   0.002561  0.025939   -0.023378   0.0 2.112806e-11         True
   total_dur_min   0.003781  0.012376   -0.008594   0.0 2.493452e-11         True
   session_count   0.009353  0.001822    0.007531 900.0 1.374212e-11         True
  answered_count   0.034723  0.041998   -0.007274   1.0 3.225564e-11         True
 completion_rate   0.000530  0.004170   -0.003640   0.0 9.264276e-12         True
   total_actions   0.007318  0.006058    0.001259 765.0 2.729238e-06         True
interaction_rate   0.000418  0.001287   -0.000868 247.5 2.184863e-03         True
  verb_diversity   0.000000  0.000456   -0.000456 195.0 1.434241e-06         True
 avg_session_dur   0.000307  0.000000    0.000307 615.0 2.852652e-04         True

Cross-subject comparison complete


In [38]:
# Descriptive Statistics
print("[Descriptive statistics]")

for feat, label in [(feat_geom, 'geometry'), (feat_geo, 'geography')]:
    desc = feat[FEATURE_COLS + ['mean_score']].describe().round(2)
    desc.to_csv(f'outputs/descriptive_{label}.csv')
    print(f"\n{label}:\n{desc.to_string()}")

print("Descriptive statistics completed")

[Descriptive statistics]

geometry:
       session_count  total_dur_min  avg_session_dur  total_actions  interaction_rate  verb_diversity  completion_count  completion_rate  answered_count  mean_score
count         677.00         677.00           677.00         677.00            677.00          677.00            677.00           677.00          677.00       677.0
mean            3.03       32023.55            34.88         210.56              0.59            3.93              8.22             0.03           79.18         0.0
std             2.72       51442.32            55.59         267.48              0.34            1.58             19.23             0.03          186.77         0.0
min             1.00           0.09             0.00           2.00              0.00            1.00              0.00             0.00            0.00         0.0
25%             1.00          18.61            11.30          46.00              0.26            3.00              0.00             0.00   

In [40]:
# Additional Figures
print("[Additions (cluster scatter + ROC curves)]")


from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc as sk_auc

# Colours for extra figures
ORANGE_DARK = COLOR_GEOM
ORANGE_LITE = '#F4C9A8'
GREEN_DARK = COLOR_GEO
GREEN_LITE = '#A8D5B5'
GREY_LINE = '#AAAAAA'


# PCA (cluster/scatter)
def plot_cluster_scatter(feat, label, engaged_col, disengaged_col, sil_score):
    X = feat[FEATURE_COLS].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X2 = pca.fit_transform(X_scaled)
    var1 = pca.explained_variance_ratio_[0] * 100
    var2 = pca.explained_variance_ratio_[1] * 100

    engaged = feat['engagement'] == 'engaged'
    disengaged = feat['engagement'] == 'disengaged'

    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.scatter(X2[disengaged, 0], X2[disengaged, 1],
               c=disengaged_col, s=18, alpha=0.55, linewidths=0,
               label=f'Disengaged (n={disengaged.sum()})')
    ax.scatter(X2[engaged, 0], X2[engaged, 1],
               c=engaged_col, s=32, alpha=0.90, linewidths=0,
               label=f'Engaged (n={engaged.sum()})')
    ax.set_xlabel(f'PC1 ({var1:.1f}% variance)', fontsize=12)
    ax.set_ylabel(f'PC2 ({var2:.1f}% variance)', fontsize=12)
    ax.set_title(f'{label}: K-Means Clusters (k\u00a0=\u00a02)\nPCA Projection of 9 Behavioural Features',
                 fontsize=13, fontweight='bold', pad=12)
    legend = ax.legend(frameon=True, framealpha=0.9, fontsize=11, loc='upper right')
    legend.get_frame().set_edgecolor('#DDDDDD')
    ax.text(0.02, 0.02, f'Silhouette: {sil_score:.3f}',
            transform=ax.transAxes, fontsize=10, va='bottom', color='#555555',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#CCCCCC', alpha=0.8))
    plt.tight_layout()
    fname = f'outputs/fig_cluster_scatter_{label.lower()}.png'
    plt.savefig(fname, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {fname}')


print("\nGenerating cluster scatter plots ...")
plot_cluster_scatter(feat_geom, 'Geometry',  ORANGE_DARK, ORANGE_LITE, sil_geom)
plot_cluster_scatter(feat_geo,  'Geography', GREEN_DARK,  GREEN_LITE,  sil_geo)


# ROC curves
def compute_roc_oof(feat, label):
    X = feat[FEATURE_COLS].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    y = (feat['engagement'] == 'disengaged').astype(int).values
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    results = {}
    for name, clf in [
        ('Logistic Regression',
         LogisticRegression(max_iter=1000, class_weight='balanced',
                            C=1.0, random_state=RANDOM_STATE)),
        ('Random Forest',
         RandomForestClassifier(n_estimators=200, max_depth=6,
                                class_weight='balanced',
                                random_state=RANDOM_STATE)),
    ]:
        oof_prob = np.zeros(len(y))
        for train_idx, test_idx in cv.split(X_scaled, y):
            clf.fit(X_scaled[train_idx], y[train_idx])
            oof_prob[test_idx] = clf.predict_proba(X_scaled[test_idx])[:, 1]
        fpr, tpr, _ = roc_curve(y, oof_prob)
        roc_auc = sk_auc(fpr, tpr)
        results[name] = (fpr, tpr, roc_auc)
        print(f'  {label} {name}: AUC = {roc_auc:.3f}')
    return results, y


def plot_roc_single(results, y, label, main_color, save_path):
    fig, ax = plt.subplots(figsize=(6, 5.5))
    styles = {
        'Logistic Regression': ('--', main_color, 2.0),
        'Random Forest':       ('-',  main_color, 2.5),
    }
    for name, (fpr, tpr, roc_auc) in results.items():
        ls, col, lw = styles[name]
        ax.plot(fpr, tpr, linestyle=ls, color=col, linewidth=lw,
                label=f'{name} (AUC\u00a0=\u00a0{roc_auc:.3f})')
        ax.fill_between(fpr, tpr, alpha=0.06, color=col)
    ax.plot([0, 1], [0, 1], linestyle=':', color=GREY_LINE, linewidth=1.5,
            label='Baseline (AUC\u00a0=\u00a00.500)')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'{label}: ROC Curves\nLogistic Regression vs Random Forest',
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    legend = ax.legend(frameon=True, framealpha=0.9, fontsize=10, loc='lower right')
    legend.get_frame().set_edgecolor('#DDDDDD')
    pct = (y == 1).mean() * 100
    ax.text(0.02, 0.02, f'Disengaged class: {pct:.1f}%',
            transform=ax.transAxes, fontsize=9, va='bottom', color='#555555',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#CCCCCC', alpha=0.8))
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {save_path}')


print("\nComputing ROC curves from cross-validation (may take ~60 seconds) ...")
roc_geom_data, y_geom_roc = compute_roc_oof(feat_geom, 'Geometry')
roc_geo_data,  y_geo_roc  = compute_roc_oof(feat_geo,  'Geography')

print("\nPlotting ROC curves ...")
plot_roc_single(roc_geom_data, y_geom_roc, 'Geometry',  ORANGE_DARK, 'outputs/fig_roc_geometry.png')
plot_roc_single(roc_geo_data,  y_geo_roc,  'Geography', GREEN_DARK,  'outputs/fig_roc_geography.png')

# Combined next to eachother
print("  Plotting combined ROC figure ...")
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, results, y_r, label, main_color in [
    (axes[0], roc_geom_data, y_geom_roc, 'Geometry',  ORANGE_DARK),
    (axes[1], roc_geo_data,  y_geo_roc,  'Geography', GREEN_DARK),
]:
    styles = {
        'Logistic Regression': ('--', main_color, 2.0),
        'Random Forest':       ('-',  main_color, 2.5),
    }
    for name, (fpr, tpr, roc_auc) in results.items():
        ls, col, lw = styles[name]
        ax.plot(fpr, tpr, linestyle=ls, color=col, linewidth=lw,
                label=f'{name} (AUC\u00a0=\u00a0{roc_auc:.3f})')
        ax.fill_between(fpr, tpr, alpha=0.07, color=col)
    ax.plot([0, 1], [0, 1], linestyle=':', color=GREY_LINE, linewidth=1.5,
            label='Baseline (AUC\u00a0=\u00a00.500)')
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    legend = ax.legend(frameon=True, framealpha=0.9, fontsize=9, loc='lower right')
    legend.get_frame().set_edgecolor('#DDDDDD')
    pct = (y_r == 1).mean() * 100
    ax.text(0.02, 0.02, f'Disengaged: {pct:.1f}%',
            transform=ax.transAxes, fontsize=9, va='bottom', color='#555555',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#CCCCCC', alpha=0.8))
fig.suptitle('ROC Curves — Logistic Regression vs Random Forest',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/fig_roc_combined.png', bbox_inches='tight')
plt.close()
print('  Saved: outputs/fig_roc_combined.png')

print("""
Saved to outputs

CSVs:
  features_geometry/geography.csv
  test_scores_geometry/geography.csv
  validation_summary.csv
  cluster_profile_geometry/geography.csv
  performance_geometry/geography.csv
  importance_geometry/geography.csv
  importance_raw_geometry/geography.csv
  cross_subject_comparison.csv
  descriptive_geometry/geography.csv

Figures:
  fig_elbow_geometry/geography.png
  fig_gain_geometry/geography.png
  fig_importance_geometry/geography.png            
  fig_cross_subject.png
  fig_cluster_scatter_geometry/geography.png    
  fig_roc_geometry/geography.png                          
  fig_roc_combined.png               

""")

[Additions (cluster scatter + ROC curves)]

Generating cluster scatter plots ...
  Saved: outputs/fig_cluster_scatter_geometry.png
  Saved: outputs/fig_cluster_scatter_geography.png

Computing ROC curves from cross-validation (may take ~60 seconds) ...
  Geometry Logistic Regression: AUC = 1.000
  Geometry Random Forest: AUC = 0.998
  Geography Logistic Regression: AUC = 1.000
  Geography Random Forest: AUC = 0.996

Plotting ROC curves ...
  Saved: outputs/fig_roc_geometry.png
  Saved: outputs/fig_roc_geography.png
  Plotting combined ROC figure ...
  Saved: outputs/fig_roc_combined.png

Saved to outputs

CSVs:
  features_geometry/geography.csv
  test_scores_geometry/geography.csv
  validation_summary.csv
  cluster_profile_geometry/geography.csv
  performance_geometry/geography.csv
  importance_geometry/geography.csv
  importance_raw_geometry/geography.csv
  cross_subject_comparison.csv
  descriptive_geometry/geography.csv

Figures:
  fig_elbow_geometry/geography.png
  fig_gain_geometr

In [42]:
# Boxplot
print("\nGenerating box plot separation figure ...")

from sklearn.preprocessing import StandardScaler as _SS

def plot_separation_boxplots(feat_g, feat_geo, feature_cols):
    """
    box plot grid, one row per feature, geometry left geography right
    z-scored for same scale
    """
    n_feats = len(feature_cols)

    # Z-scores
    sc = _SS()
    Xg = sc.fit_transform(feat_g[feature_cols].fillna(0))
    sc2 = _SS()
    Xge = sc2.fit_transform(feat_geo[feature_cols].fillna(0))

    fig, axes = plt.subplots(n_feats, 2, figsize=(11, n_feats * 1.35))
    fig.subplots_adjust(hspace=0.55, wspace=0.35)

    for feat_idx, feat_name in enumerate(feature_cols):
        for col_idx, (X, feat_df, color, label) in enumerate([
            (Xg,  feat_g,   ORANGE_DARK, 'Geometry'),
            (Xge, feat_geo, GREEN_DARK,  'Geography'),
        ]):
            ax = axes[feat_idx, col_idx]

            engaged    = X[feat_df['engagement'] == 'engaged',    feat_idx]
            disengaged = X[feat_df['engagement'] == 'disengaged', feat_idx]

            bp = ax.boxplot(
                [engaged, disengaged],
                patch_artist=True,
                widths=0.5,
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(color=color, linewidth=1.2),
                capprops=dict(color=color, linewidth=1.2),
                flierprops=dict(marker='o', markersize=2,
                                markerfacecolor=color, alpha=0.3,
                                linestyle='none'),
                notch=False,
            )

            # Lighter/Darker
            bp['boxes'][0].set_facecolor(color)
            bp['boxes'][0].set_alpha(0.85)
            bp['boxes'][1].set_facecolor(color)
            bp['boxes'][1].set_alpha(0.30)

            ax.set_xticks([1, 2])
            ax.set_xticklabels(['Engaged', 'Disengaged'], fontsize=8.5)
            ax.set_ylabel('z-score', fontsize=8)
            ax.tick_params(axis='y', labelsize=7.5)
            ax.axhline(0, color='#CCCCCC', linewidth=0.6, linestyle='--')

            # Feature name on left column only
            if col_idx == 0:
                ax.set_title(f'{feat_name}  |  {label}',
                             fontsize=9, fontweight='bold', loc='left', pad=3)
            else:
                ax.set_title(label, fontsize=9, loc='left', pad=3)

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

    fig.suptitle(
        'Engaged vs Disengaged — Feature Distributions\n'
        '(z-scored; darker = engaged, lighter = disengaged)',
        fontsize=13, fontweight='bold', y=1.005
    )
    plt.savefig('outputs/fig_separation_boxplots.png',
                dpi=150, bbox_inches='tight')
    plt.close()
    print('  Saved: outputs/fig_separation_boxplots.png')


plot_separation_boxplots(feat_geom, feat_geo, FEATURE_COLS)


Generating box plot separation figure ...
  Saved: outputs/fig_separation_boxplots.png


In [58]:
# Answered & session count
print("\nGenerating two-feature cluster scatter ...")


def plot_two_feature_scatter(feat_g, feat_geo):
    """
    Scatter Plot of the Two Strongest Features for Classifier Visualisation
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

    for ax, feat_df, color, title in [
        (axes[0], feat_g,   ORANGE_DARK, 'Geometry'),
        (axes[1], feat_geo, GREEN_DARK,  'Geography'),
    ]:
        engaged    = feat_df[feat_df['engagement'] == 'engaged']
        disengaged = feat_df[feat_df['engagement'] == 'disengaged']

        ax.scatter(
            disengaged['answered_count'],
            disengaged['session_count'],
            c=color, alpha=0.30, s=20, linewidths=0,
            label=f'Disengaged (n={len(disengaged)})'
        )
        ax.scatter(
            engaged['answered_count'],
            engaged['session_count'],
            c=color, alpha=0.90, s=45, linewidths=0,
            marker='D',
            label=f'Engaged (n={len(engaged)})'
        )

        ax.set_xlabel('answered_count (quiz attempts)', fontsize=11)
        ax.set_ylabel('session_count (active days)',    fontsize=11)
        ax.set_title(title, fontsize=13, fontweight='bold')

        legend = ax.legend(frameon=True, framealpha=0.9,
                           fontsize=10, loc='upper right')
        legend.get_frame().set_edgecolor('#DDDDDD')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Region annotations
        ax.text(0.97, 0.05,
                'Low quiz attempts\nfew sessions',
                transform=ax.transAxes, fontsize=9,
                ha='right', va='bottom', color='#888888', style='italic')
        ax.text(0.97, 0.95,
                'High quiz attempts\nmany sessions',
                transform=ax.transAxes, fontsize=9,
                ha='right', va='top', color=color, style='italic')

    fig.suptitle(
        'K-Means Cluster Separation — Engaged vs Disengaged\n'
        'Plotted on the two most predictive features',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig('outputs/fig_cluster_two_feature.png',
                dpi=150, bbox_inches='tight')
    plt.close()
    print('  Saved: outputs/fig_cluster_two_feature.png')


plot_two_feature_scatter(feat_geom, feat_geo)

print("\nTwo feature cluster completed")



Generating two-feature cluster scatter ...
  Saved: outputs/fig_cluster_two_feature.png

Two feature cluster completed
